In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from skimage.transform import resize
from sklearn.utils import shuffle

# Suppress TensorFlow warnings for cleaner output
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# ===============================
# 1. Configuration and Data Paths
# ===============================

# Use your dataset's name and class count
DATASET_NAME = 'brain-tumor-mri-dataset'
NUM_CLASSES = 4
IMG_HEIGHT, IMG_WIDTH = 224, 224
BATCH_SIZE = 32
EPOCHS = 12

# Correctly define paths for a Kaggle environment
# train_dir = f'/kaggle/input/{DATASET_NAME}/Training/'
# test_dir = f'/kaggle/input/{DATASET_NAME}/Testing/'
train_dir = r"e:\HealthCareProject\DeepLearning_module\notebook\img_data\brain_tumor_mri_dataset\Training"
test_dir = r"e:\HealthCareProject\DeepLearning_module\notebook\img_data\brain_tumor_mri_dataset\Testing"

# ===============================
# 2. Data Preprocessing & Generation (Using the same method as your notebook)
# ===============================

labels = ['glioma', 'meningioma', 'notumor', 'pituitary']

X_train_raw = []  # Training Dataset
Y_train_raw = []  # Training Labels

# Load images from the training directory
for label in labels:
    path = os.path.join(train_dir, label)
    class_num = labels.index(label)
    for img_name in os.listdir(path):
        img_path = os.path.join(path, img_name)
        img_array = plt.imread(img_path)
        img_resized = resize(img_array, (IMG_HEIGHT, IMG_WIDTH, 3))
        X_train_raw.append(img_resized)
        Y_train_raw.append(class_num)

# Load images from the testing directory
for label in labels:
    path = os.path.join(test_dir, label)
    class_num = labels.index(label)
    for img_name in os.listdir(path):
        img_path = os.path.join(path, img_name)
        img_array = plt.imread(img_path)
        img_resized = resize(img_array, (IMG_HEIGHT, IMG_WIDTH, 3))
        X_train_raw.append(img_resized)
        Y_train_raw.append(class_num)

# Convert to NumPy arrays
X_train_raw = np.array(X_train_raw)
Y_train_raw = np.array(Y_train_raw)

# Shuffle the data
X_train_shuffled, Y_train_shuffled = shuffle(X_train_raw, Y_train_raw, random_state=42)

# Split the data into training, validation, and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(X_train_shuffled, Y_train_shuffled, test_size=0.2, random_state=42)
X_train, X_valid, Y_train, Y_valid = train_test_split(X_train, Y_train, test_size=0.1, random_state=42)

# Convert labels to categorical format (one-hot encoding)
Y_train_new = to_categorical(Y_train, num_classes=NUM_CLASSES)
Y_valid_new = to_categorical(Y_valid, num_classes=NUM_CLASSES)
Y_test_new = to_categorical(Y_test, num_classes=NUM_CLASSES)

# Use ImageDataGenerator for on-the-fly augmentation on the training set
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,  # Rescale pixel values
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest"
)

# Fit the generator on the training data
train_datagen.fit(X_train)

# ===============================
# 3. Model Building (VGG16 Transfer Learning)
# ===============================

# Load the pre-trained VGG16 model without the final classification layers
base_model = VGG16(weights="imagenet", include_top=False, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))

# Freeze the layers of the base model
for layer in base_model.layers:
    layer.trainable = False

# Build a new classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)
predictions = Dense(NUM_CLASSES, activation="softmax")(x)

# Create the final model
model = Model(inputs=base_model.input, outputs=predictions)

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss="categorical_crossentropy",
              metrics=["accuracy"])

# Callbacks for early stopping and model checkpoint
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
checkpoint = tf.keras.callbacks.ModelCheckpoint('Bmodel_vgg16.h5', monitor='val_loss', save_best_only=True)

# Model summary
model.summary()

# ===============================
# 4. Phase 1: Training (Feature Extraction)
# ===============================

print("\n--- Starting Phase 1: Feature Extraction (Frozen Base) ---")
history = model.fit(
    train_datagen.flow(X_train, Y_train_new, batch_size=BATCH_SIZE),
    validation_data=(X_valid, Y_valid_new),
    epochs=10,
    callbacks=[early_stopping, checkpoint]
)

# ===============================
# 5. Phase 2: Fine-Tuning
# ===============================

# Unfreeze the last few convolutional layers of the VGG16 model for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-4]: # Fine-tune the last 4 convolutional layers
    layer.trainable = False

# Recompile the model with a very low learning rate for fine-tuning
model.compile(optimizer=Adam(learning_rate=0.00001),
              loss="categorical_crossentropy",
              metrics=["accuracy"])

print("\n--- Starting Phase 2: Fine-Tuning ---")
history_fine_tune = model.fit(
    train_datagen.flow(X_train, Y_train_new, batch_size=BATCH_SIZE),
    validation_data=(X_valid, Y_valid_new),
    epochs=20, # Use more epochs for this phase
    initial_epoch=history.epoch[-1],
    callbacks=[early_stopping, checkpoint]
)


# ===============================
# 6. Evaluation
# ===============================

# Load the best model saved by the checkpoint
model = tf.keras.models.load_model('Bmodel_vgg16.h5')

print("\n--- Final Evaluation ---")
loss, acc = model.evaluate(X_test, Y_test_new, verbose=1)
print(f"Final Test Accuracy: {acc:.4f}")

# Get predictions for detailed metrics
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(Y_test_new, axis=1)
class_labels = labels

print("\nClassification Report:")
print(classification_report(y_true, y_pred_classes, target_names=class_labels))

# ===============================
# 7. Visualization
# ===============================

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_classes)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# Training History Plot
plt.figure(figsize=(12,5))
combined_history_acc = history.history["accuracy"] + history_fine_tune.history["accuracy"]
combined_history_val_acc = history.history["val_accuracy"] + history_fine_tune.history["val_accuracy"]

plt.subplot(1,2,1)
plt.plot(combined_history_acc, label="Train Acc")
plt.plot(combined_history_val_acc, label="Val Acc")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Accuracy Over Epochs")

combined_history_loss = history.history["loss"] + history_fine_tune.history["loss"]
combined_history_val_loss = history.history["val_loss"] + history_fine_tune.history["val_loss"]

plt.subplot(1,2,2)
plt.plot(combined_history_loss, label="Train Loss")
plt.plot(combined_history_val_loss, label="Val Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss Over Epochs")

plt.show()

MemoryError: Unable to allocate 6.30 GiB for an array with shape (5618, 224, 224, 3) and data type float64